# v5无旧上限五会话锁定：完整分钟来源与整股支持

事前冻结 `plan.json`。以逐层点时父本、双ETF Qlib 全分钟源和已保存的次开盘整股掩码，分开计数决策前意向、之后整股成交、五会话路径成熟。恢复前/后状态按实际结算时刻，10:30–10:32次序未知。只读来源与费用，不读取未来动作价值。

In [1]:
import os,json,hashlib,warnings
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_DISABLE_AGENT_HINT']='1'
warnings.filterwarnings('ignore',message="The 'generic' unit for NumPy timedelta")
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
from src.analysis.stage1_capfree_lock_support import audit_capfree_lock_support
import exchange_calendars as xc
root=Path.cwd();card=root/'research/stage1_v5_capfree_lock_fullminute_support_20261030';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan['minute_provider_uri']/'qlib_store_manifest.json')==plan['minute_provider_manifest_sha256']
assert sha(root/plan['lock_module'])==plan['lock_module_sha256']
for role in ('parent','frame','saved_next_open_labels'):
 s=plan[role];source=root/s['card'];assert sha(source/'summary.json')==s['summary_sha256'] and sha(source/'verification_manifest.json')==s['manifest_sha256']
 assert json.loads((source/'nas_binding.json').read_text())['evidence']['sha256']==s['nas_archive_sha256']
label_plan_path=root/plan['saved_next_open_labels']['card']/'plan.json';assert sha(label_plan_path)==plan['saved_label_plan_sha256']
qlib.init(provider_uri=str(root/plan['minute_provider_uri']),region='us',kernels=1)
def read(role,name):
 s=plan[role];v,_=load_verified_recorder_objects(root/s['card'],expected_recorder_id=s['recorder_id'],artifacts=[name],expected_manifest_sha256=s['manifest_sha256']);return v[name]
book=read('parent','analysis/native/dated_f696_0')
frame=read('frame','inputs/covered_frame').reset_index(drop=True)
saved=read('saved_next_open_labels','analysis/labels')
assert len(frame)==len(book['independent_replay'])==plan['parent_sessions'] and len(saved)==384660
assert frame.trade_date.equals(book['independent_replay'].trade_date.reset_index(drop=True))
tariff=json.loads(label_plan_path.read_text())['tariff']
print('verified Qlib Recorder parent/frame/saved-support identities')

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[25250:MainThread](2026-10-30 13:51:57,059) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[25250:MainThread](2026-10-30 13:51:57,471) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[25250:MainThread](2026-10-30 13:51:57,471) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


verified Qlib Recorder parent/frame/saved-support identities


In [2]:
raw=D.features(['SOXL','SOXX'],['$open','$close','$raw_open','$price_valid'],start_time=plan['start']+' 09:31:00',end_time=plan['end']+' 16:00:00',freq='1min')
parts=[]
for symbol in ('SOXL','SOXX'):
 x=raw.xs(symbol,level='instrument').copy();x.columns=[symbol+'_'+c.removeprefix('$') for c in x.columns];parts.append(x)
assert parts[0].index.equals(parts[1].index)
quotes=pd.concat(parts,axis=1).sort_index();assert quotes.index.is_unique
schedule=xc.get_calendar('XNYS',start=frame.trade_date.min(),end=frame.trade_date.max()).schedule
for day,part in quotes.groupby(quotes.index.normalize(),sort=True):
 expected=pd.Timestamp(schedule.loc[day]['close']).tz_convert('America/New_York').tz_localize(None)
 assert part.index.max()==expected
print('Qlib minute rows',len(raw),'paired days',quotes.index.normalize().nunique())

Qlib minute rows 820440 paired days 1056


In [3]:
perday=audit_capfree_lock_support(frame,book,quotes,saved,tariff=tariff,delta_beta=.10,horizon_sessions=5)
assert len(perday)==1056
sumcols=['total_clock_slots','ambiguous_clock','decision_source_valid','cash_intents','fill_source_valid','mature_path_source_intents','soxl_funded','soxx_funded','paired_funded','no_whole_share_pair','saved_funding_reused','newly_settled_funding']
counts={k:int(perday[k].sum()) for k in sumcols}
annual=[]
for year,g in perday.groupby('year'):
 annual.append({'year':int(year),'sessions':len(g),'mature_sessions':int(g.five_session_mature.sum()),'days_with_cash_intent':int(g.cash_intents.gt(0).sum()),'days_with_mature_path_source_intent':int(g.mature_path_source_intents.gt(0).sum()),'days_with_paired_funded_path':int(g.paired_funded.gt(0).sum()),'paired_funded_rows':int(g.paired_funded.sum()),'ambiguous_clock_rows':int(g.ambiguous_clock.sum())})
# Five-session starts overlap. Greedy earliest possible calendar starts give a
# source-only upper bound on non-overlapping labels, not an observed policy.
selected=[];next_allowed=0
for i,row in perday.iterrows():
 if i>=next_allowed and row.paired_funded>0:
  selected.append((i,int(row.year)));next_allowed=i+5
disjoint={str(year):int(sum(y==year for _,y in selected)) for year in sorted(perday.year.unique())}
gates=plan['support_gates'];by_year={x['year']:x for x in annual};discovery=sum(by_year[y]['days_with_paired_funded_path'] for y in (2018,2019,2020))
support_pass=discovery>=gates['minimum_discovery_days_2018_2020'] and all(by_year[y]['days_with_paired_funded_path']>=gates['minimum_'+str(y)+'_days'] for y in (2019,2020,2021,2022))
perday.to_csv(card/'perday_support.csv',index=False)
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'support_code_sha256':sha(root/'src/analysis/stage1_capfree_lock_support.py'),'parent_recorder_id':plan['parent']['recorder_id'],'source_rows':len(raw),'parent_sessions':len(frame),'counts':counts,'annual':annual,'nonoverlap_earliest_start_upper_bound':{'total':len(selected),'by_year':disjoint},'source_support_gate_pass':bool(support_pass),'new_model_fits':0,'new_candidate_accounts':0,'new_recorders':0,'future_value_columns_accessed':False,'historical_research_passed':False,'decision':'fullminute_five_session_source_support_pass_not_information_or_account' if support_pass else 'fullminute_five_session_source_support_failed'}
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(audit,ensure_ascii=False,indent=2))

{
  "experiment": "stage1_v5_capfree_lock_fullminute_support_20261030",
  "plan_sha256": "912cd2767ba084b8a4e8a80fb5d8c5c048f7e488294a6383e1d385705c7d32e6",
  "support_code_sha256": "bccecf404abd36c960cba1edbdb804d9584f579688a526c4e73de0aaf88ca88a",
  "parent_recorder_id": "2994f6afbfc54f279cefec362b220001",
  "source_rows": 820440,
  "parent_sessions": 1056,
  "counts": {
    "total_clock_slots": 389100,
    "ambiguous_clock": 3168,
    "decision_source_valid": 385932,
    "cash_intents": 385932,
    "fill_source_valid": 385932,
    "mature_path_source_intents": 382996,
    "soxl_funded": 382489,
    "soxx_funded": 332757,
    "paired_funded": 332277,
    "no_whole_share_pair": 50719,
    "saved_funding_reused": 379693,
    "newly_settled_funding": 3303
  },
  "annual": [
    {
      "year": 2018,
      "sessions": 53,
      "mature_sessions": 53,
      "days_with_cash_intent": 53,
      "days_with_mature_path_source_intent": 53,
      "days_with_paired_funded_path": 53,
      "paired

## 中文结论

**SOXX：未运行信息模型或候选账户，0/2/5bp全门未测。** Qlib双ETF分钟共820440行覆盖1056父本会话；389100个理论因果钟点中，10:30–10:32先后不明3168个，剩385932个决策前已知且有固定0.10现金意向。未来填单与五会话来源路径合格382996个，其中332277个两臂均按原价整股Tiered形成买单、50719个未形成完整配对整股。不能把funded/path掩码当实时模型输入。

按年配对完整路径日为2018部分53、2019 252、2020 249、2021 252、2022 242，旧来源逐年支持门通过。2020-03-10至13的五会话路径经过03-16一根无效价格bar而标未知；2022-12-19至22缺五会话终端。五会话标签高度重叠，按最早来源可行日排出的不重叠起点最多211个，分年11/50/51/50/49；这只是来源上界，不是统计独立观测数或策略成交天数。旧标签funded标志复用379693条，新风险时钟下重新整股核3303条，2021/2022的价值列没有进入计算。

第一次执行在完成Qlib逐日表后因`numpy.int64` JSON写出失败；保存原计划和收据，只改序列化并按计划补记第二次来源读取，结果通过。下一步先以此真实支持/重叠规模制定计算预算和时间分组，再生成同域跨会话数值标签；完整分钟意向在未来填单前评分，锁定期不重复触发，预测→整股→账户NAV仍未证明。